# SynthScreen: AI Biodesign Guardrails — Colab Training Notebook
**AIxBio Hackathon 2026 — Track 1: DNA Screening & Synthesis Controls**

## What we're building
- **Problem**: ProteinMPNN/RFdiffusion can design functional toxins that are sequence-divergent from known hazards → BLAST-based screening misses them
- **Solution**: DNABERT-2 + ESM-2 fine-tuned to detect *functional* hazard, not just sequence similarity
- **Key result**: We demonstrate the gap — show sequences that BLAST misses, SynthScreen catches

## Pipeline
1. Install dependencies
2. Clone SynthScreen + ProteinMPNN
3. Build dataset (NCBI + ProteinMPNN variants)
4. Train DNABERT-2 (DNA track)
5. Train ESM-2 (protein track)
6. Benchmark vs BLAST
7. Save to HuggingFace Hub

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch Runtime to GPU!"}')
print(f'CUDA: {torch.version.cuda}')
print(f'PyTorch: {torch.__version__}')
!nvidia-smi | head -12

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install transformers==4.40.0 peft==0.10.0 datasets==2.19.0 accelerate==0.29.0
!pip install biopython scikit-learn matplotlib seaborn gradio tqdm huggingface_hub
!pip install bitsandbytes  # 8-bit quantization for ESM-2
print('Done installing')

## Step 2: Clone SynthScreen Repo & ProteinMPNN

In [ ]:
import os

# Clone SynthScreen
if not os.path.exists('synthscreen'):
    !git clone https://github.com/Ashok-kumar290/synthscreen.git
os.chdir('synthscreen')
print('Working dir:', os.getcwd())

# Clone ProteinMPNN
if not os.path.exists('ProteinMPNN'):
    !git clone https://github.com/dauparas/ProteinMPNN.git
    !pip install -q pyrosetta-installer 2>/dev/null; true

print('Ready.')

## Step 3: Configuration

In [ ]:
# ── SET THESE ──
NCBI_EMAIL = 'your@email.com'   # CHANGE THIS
NCBI_API_KEY = ''               # Optional: get at https://www.ncbi.nlm.nih.gov/account/
HF_TOKEN = ''                   # Optional: your HuggingFace token for saving models
HF_REPO = 'Ashok-kumar290/synthscreen-dnabert2'  # Where to push the model

# Training settings
MAX_PER_QUERY = 300       # Sequences per NCBI query
N_MPNN_SEQS = 50          # ProteinMPNN sequences per structure
BATCH_SIZE = 16           # Increase if A100 (use 32)
EPOCHS = 6
LEARNING_RATE = 2e-4

print('Config set.')
print(f'Training: {EPOCHS} epochs, batch={BATCH_SIZE}, lr={LEARNING_RATE}')

## Step 4A: Build DNA Dataset from NCBI
Downloads hazardous and benign sequences, generates codon-shuffled variants (simulating AI-redesigned sequences)

In [ ]:
!python data/build_dataset.py \
    --email {NCBI_EMAIL} \
    --api_key {NCBI_API_KEY} \
    --output data/processed/synthscreen_dna_v1_dataset \
    --max_per_query {MAX_PER_QUERY}

In [ ]:
from datasets import load_from_disk
ds = load_from_disk('data/processed/synthscreen_dna_v1_dataset')
print(ds)
print('\nSample example:')
ex = ds['train'][0]
print(f'  Label: {ex["label"]} (1=hazardous, 0=benign)')
print(f'  Source: {ex.get("source", "unknown")}')
print(f'  Sequence length: {len(ex["sequence"])} bp')
print(f'  Sequence: {ex["sequence"][:80]}...')

# Class balance check
labels = ds['train']['label']
n_haz = sum(labels)
n_ben = len(labels) - n_haz
print(f'\nTrain class balance: {n_haz} hazardous / {n_ben} benign')

## Step 4B: Generate ProteinMPNN Variants (Novel Contribution)
This generates the sequences that demonstrate the BLAST screening gap:
- Take known dangerous protein structures (Ricin, BoNT, Anthrax LF, etc.)
- Design novel sequences via ProteinMPNN
- Back-translate to DNA with codon optimization
- These sequences PASS BLAST but FAIL SynthScreen

In [ ]:
!python data/generate_mpnn_variants.py \
    --n_seqs {N_MPNN_SEQS} \
    --output data/mpnn_variants.json \
    --mpnn_dir ProteinMPNN

In [ ]:
import json
with open('data/mpnn_variants.json') as f:
    mpnn_data = json.load(f)

print('ProteinMPNN generation summary:')
print(json.dumps(mpnn_data['summary'], indent=2))

# Show identity distribution
if mpnn_data['dangerous']:
    ids = [e['blast_identity_estimate'] for e in mpnn_data['dangerous']]
    import numpy as np
    print(f'\nBlAST identity estimates for dangerous variants:')
    print(f'  Mean: {np.mean(ids):.1%}')
    print(f'  Median: {np.median(ids):.1%}')
    print(f'  < 50% identity (BLAST-evasive): {sum(1 for x in ids if x < 0.5)} / {len(ids)}')
    print(f'  < 70% identity (below common threshold): {sum(1 for x in ids if x < 0.7)} / {len(ids)}')

## Step 5: Train DNABERT-2 (DNA Track)

In [ ]:
!python scripts/training/train_synthscreen.py \
    --config configs/synthscreen_v1.json \
    --model_type dnabert2 \
    --dataset_path data/processed/synthscreen_dna_v1_dataset \
    --output_dir models/synthscreen_dnabert2 \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --learning_rate {LEARNING_RATE}

## Step 5B: Train ESM-2 (Protein Track)
For screening at the amino acid level — catches even more AI-designed variants.
Requires a protein sequence dataset. Using funcscreen's existing dataset if available.

In [ ]:
# Build a protein-level dataset from ProteinMPNN variants + original sequences
import json
from datasets import Dataset, DatasetDict
import random

with open('data/mpnn_variants.json') as f:
    mpnn_data = json.load(f)

protein_examples = []
for entry in mpnn_data.get('dangerous', []):
    seq = entry.get('protein_sequence', '')
    if seq:
        protein_examples.append({'sequence': seq, 'label': 1, 'source': 'mpnn_dangerous'})

for entry in mpnn_data.get('benign', []):
    seq = entry.get('protein_sequence', '')
    if seq:
        protein_examples.append({'sequence': seq, 'label': 0, 'source': 'mpnn_benign'})

random.shuffle(protein_examples)
n = len(protein_examples)
n_train = int(0.7 * n)
n_val = int(0.15 * n)

if n >= 10:
    protein_ds = DatasetDict({
        'train': Dataset.from_list(protein_examples[:n_train]),
        'validation': Dataset.from_list(protein_examples[n_train:n_train+n_val]),
        'test': Dataset.from_list(protein_examples[n_train+n_val:]),
    })
    protein_ds.save_to_disk('data/processed/synthscreen_protein_v1_dataset')
    print(f'Saved protein dataset: {n} examples')
    print(protein_ds)
else:
    print(f'Only {n} examples — need more ProteinMPNN variants. Increase N_MPNN_SEQS.')

In [ ]:
import os
if os.path.exists('data/processed/synthscreen_protein_v1_dataset'):
    !python scripts/training/train_synthscreen.py \
        --model_type esm2 \
        --dataset_path data/processed/synthscreen_protein_v1_dataset \
        --output_dir models/synthscreen_esm2 \
        --epochs {EPOCHS} \
        --batch_size 8 \
        --max_seq_length 512
else:
    print('Protein dataset not found. Run the cell above first.')

## Step 6: Benchmark — SynthScreen vs BLAST
This is the **key result** for the hackathon submission:
- ProteinMPNN-designed dangerous variants: BLAST detection rate vs SynthScreen detection rate
- Shows the gap in current AI-era DNA synthesis screening

In [ ]:
import os
model_dir = 'models/synthscreen_dnabert2/best'
if not os.path.exists(model_dir):
    # Try alternate path
    model_dir = 'models/synthscreen_dnabert2'
    
!python scripts/eval/benchmark_vs_blast.py \
    --model_dir {model_dir} \
    --model_type dnabert2 \
    --variants_json data/mpnn_variants.json \
    --blast_threshold 0.70 \
    --output_dir results/benchmark

In [ ]:
import json
from IPython.display import Image, display

with open('results/benchmark/benchmark_results.json') as f:
    results = json.load(f)

print('='*60)
print('KEY RESULT: SynthScreen vs BLAST')
print('='*60)
print(f'\nOn {results["n_dangerous_variants"]} ProteinMPNN-designed dangerous sequences:')
print(f'  SynthScreen detection:  {results["synthscreen"]["pct_caught"]}%')
print(f'  BLAST detection:        {results["blast"]["pct_caught"]}%')
print(f'  BLAST miss rate:        {results["blast"]["pct_missed"]}%')
print(f'  Recall improvement:     +{results["gap_vs_blast"]["recall_improvement"]:.1%}')
print(f'  Additional sequences caught: {results["gap_vs_blast"]["additional_sequences_caught"]}')

print(f'\nFull metrics:')
print(f'  SynthScreen F1:    {results["synthscreen"]["f1"]:.3f}')
print(f'  SynthScreen AUROC: {results["synthscreen"]["auroc"]:.3f}')
print(f'  BLAST F1:          {results["blast"]["f1"]:.3f}')

display(Image('results/benchmark/benchmark_plot.png'))

## Step 7: Save Model to HuggingFace Hub

In [ ]:
if HF_TOKEN:
    from huggingface_hub import login, HfApi
    login(token=HF_TOKEN)
    
    from transformers import AutoTokenizer
    from peft import PeftModel
    import torch
    
    model_path = 'models/synthscreen_dnabert2/best'
    if os.path.exists(model_path):
        api = HfApi()
        api.create_repo(HF_REPO, exist_ok=True)
        api.upload_folder(
            folder_path=model_path,
            repo_id=HF_REPO,
            repo_type='model',
        )
        print(f'Model pushed to: https://huggingface.co/{HF_REPO}')
    else:
        print('Model directory not found.')
else:
    print('Set HF_TOKEN to push to HuggingFace Hub.')

## Step 8: Launch Gradio Demo (for judges)

In [ ]:
import os
model_dir = 'models/synthscreen_dnabert2/best'
if not os.path.exists(model_dir):
    model_dir = 'models/synthscreen_dnabert2'

# Launch with public URL for sharing with judges
!python app/demo.py --model_dir {model_dir} --model_type dnabert2 --share

## Summary: What to report

From your results, the key numbers for the hackathon report:

1. **BLAST miss rate**: % of ProteinMPNN-designed dangerous sequences that passed BLAST at 70% threshold
2. **SynthScreen recall**: % caught by our model
3. **AUROC**: SynthScreen discriminative power
4. **Short sequence performance**: Detection rate on fragments < 150bp

The narrative:
> AI biodesign tools (ProteinMPNN, RFdiffusion) create a new attack surface: 
> functional analogs of dangerous proteins that are sequence-divergent from known hazards.
> Current DNA synthesis screening (BLAST-based) misses X% of these variants.
> SynthScreen, fine-tuned DNABERT-2 with hard example mining on codon-diverse training data,
> catches Y% of these variants — closing the AI-era biosecurity gap.